# Notebook 06 — Population Dynamics and Evolutionary Simulation

**Module:** 15 — Simulation and Agent-Based Modeling  
**Tier:** 2 — Working competence  
**Notebook:** 6 of 12  
**Time estimate:** 75 minutes

> Evolution is a population-level phenomenon driven by individual-level events:
> mutation, selection, and genetic drift. This notebook implements the Wright-Fisher
> model of genetic drift, natural selection, and Hardy-Weinberg equilibrium
> from scratch, simulating the stochastic dynamics of allele frequencies.

---
## Step 1 — Motivation

Why do allele frequencies change even in populations with no selection pressure?
Genetic drift — the random sampling of alleles each generation — causes allele
frequencies to wander and eventually reach fixation (0 or 100%). In small
populations this happens fast; in large populations it's slow. This explains
why neutral mutations can spread through a population, and why bottlenecks
(e.g., founding of a new island colony) cause rapid genetic change.

---
## Step 2 — Intuition

**Hardy-Weinberg equilibrium:** in an infinite, randomly mating population
with no selection, mutation, migration, or drift, allele frequencies stay constant
forever. This is the *null model* of population genetics — deviations from HWE
indicate one of those forces is acting.

**Wright-Fisher model:** each generation of N diploid individuals is formed by
sampling 2N alleles *with replacement* from the previous generation. Because
sampling is random, allele frequencies drift. The drift is equivalent to binomial
sampling: if allele A has frequency p, the next generation's count of A alleles
is Binomial(2N, p).

**Fixation:** eventually, drift drives the allele to 0 (loss) or 1 (fixation).
- Time to fixation scales as ~4N generations for a neutral allele.
- Fixation probability of a new mutation = 1/(2N) (neutral)
- With positive selection (coefficient s): fixation probability ≈ 2s (for Ns >> 1)

**Fitness landscape:** a map from genotype (or phenotype) to reproductive fitness.
Selection moves populations uphill on the fitness landscape — but drift can move
them across valleys (genetic drift can fix a slightly deleterious mutation).

---
## Step 3 — Biological Background

**Neutral theory of molecular evolution (Kimura 1968):** most fixed differences
between species at the molecular level are neutral — they spread by drift, not
selection. This was controversial (and still partly contested) but predicts
the molecular clock correctly.

**Founder effects:** The Afrikaner population of South Africa was founded by a
small group of Dutch settlers; some rare genetic diseases (Huntington's disease,
Fanconi anaemia type A) are elevated — drift fixed rare alleles during the
bottleneck.

**Selective sweeps:** a beneficial mutation spreads rapidly through a population.
As it does, nearby neutral alleles hitchhike — producing a "valley" of low
diversity around the selected locus. These sweeps are detectable in modern
genome sequencing data (McDonald-Kreitman test, Tajima's D).

**Mutation-selection balance:** most deleterious mutations are eventually removed
by selection, but new ones keep arising. At equilibrium, allele frequency of a
deleterious allele with mutation rate μ and selection coefficient s is
$\hat{q} \approx \sqrt{\mu/s}$ (recessive) or $\mu/s$ (dominant).

---
## Step 4 — Mathematical Explanation

**Hardy-Weinberg equilibrium:**
Allele frequencies: $p$ (A), $q = 1-p$ (a).
Genotype frequencies: $p^2$ (AA), $2pq$ (Aa), $q^2$ (aa).
HWE is reached in one generation of random mating.

**Wright-Fisher model:**
$$p_{t+1} = \frac{1}{2N} \sum_{i=1}^{2N} X_i, \quad X_i \sim \text{Bernoulli}(p_t)$$
Equivalently: $k_{t+1} \sim \text{Binomial}(2N, p_t)$, $p_{t+1} = k_{t+1}/(2N)$.

**Selection (diploid):**
With fitnesses $w_{AA}=1+s$, $w_{Aa}=1+s/2$, $w_{aa}=1$ (additive selection),
the frequency after selection:
$$p' = \frac{p^2(1+s) + pq(1+s/2)}{\bar{w}}, \quad \bar{w} = p^2(1+s) + 2pq(1+s/2) + q^2$$
Then sample from Binomial(2N, p') for the next generation.

**Effective population size Ne:** the size of an ideal Wright-Fisher population
that would experience the same rate of drift as the actual population.
$\sigma^2_p \approx p(1-p)/(2N_e)$ — variance of allele frequency change per generation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import binom

rng = np.random.default_rng(42)

# ---- Hardy-Weinberg check ----
def hardy_weinberg_check(n_alleles_A, N_diploid):
    """
    Check HWE: given allele counts, test whether genotype frequencies
    match HW expectations.
    """
    total_alleles = 2 * N_diploid
    p = n_alleles_A / total_alleles
    q = 1 - p
    # Expected genotype frequencies under HWE
    exp_AA = p**2 * N_diploid
    exp_Aa = 2*p*q * N_diploid
    exp_aa = q**2 * N_diploid
    return p, {'AA': exp_AA, 'Aa': exp_Aa, 'aa': exp_aa}

p0_hw = 0.3
p_hw, hwe = hardy_weinberg_check(int(0.3 * 2 * 10000), 10000)
print(f'HWE at p=0.3: AA={hwe["AA"]:.0f}, Aa={hwe["Aa"]:.0f}, aa={hwe["aa"]:.0f}')

# ---- Wright-Fisher model from scratch ----
def wright_fisher(p0, N, n_gen, s=0.0, rng=rng):
    """
    Simulate one Wright-Fisher trajectory.
    p0: initial allele frequency of A
    N: diploid population size (2N alleles sampled per generation)
    n_gen: number of generations
    s: additive selection coefficient for A allele
    Returns: array of allele frequencies (n_gen+1,)
    """
    p = p0
    traj = [p]
    for _ in range(n_gen):
        if s != 0.0:
            q = 1 - p
            # Additive selection: w_AA=1+s, w_Aa=1+s/2, w_aa=1
            w_bar = p**2*(1+s) + 2*p*q*(1+s/2) + q**2
            p_sel = (p**2*(1+s) + p*q*(1+s/2)) / w_bar
        else:
            p_sel = p
        # Binomial sampling
        k = rng.binomial(2*N, p_sel)
        p = k / (2*N)
        traj.append(p)
        if p == 0.0 or p == 1.0:
            # Fixation or loss — extend trajectory
            traj.extend([p] * (n_gen - len(traj) + 1))
            break
    return np.array(traj[:n_gen+1])

# Run many trajectories for different population sizes
N_SIZES = [20, 100, 1000]
N_GEN   = 200
N_TRAJ  = 30
p0 = 0.5

trajectories = {}
for N_pop in N_SIZES:
    trajectories[N_pop] = [wright_fisher(p0, N_pop, N_GEN) for _ in range(N_TRAJ)]

# Fixation statistics
for N_pop in N_SIZES:
    trajs = trajectories[N_pop]
    fixed = sum(1 for t in trajs if t[-1] == 1.0)
    lost  = sum(1 for t in trajs if t[-1] == 0.0)
    print(f'N={N_pop}: fixed={fixed}/{N_TRAJ}, lost={lost}/{N_TRAJ}')

# ---- Selection coefficient effect ----
s_values = [0.0, 0.01, 0.05, 0.1]
N_sel = 200
N_TRAJ_SEL = 50
fixation_probs = {}
for s in s_values:
    trajs = [wright_fisher(1/(2*N_sel), N_sel, 2000, s=s) for _ in range(N_TRAJ_SEL)]
    fixation_probs[s] = sum(1 for t in trajs if t[-1] == 1.0) / N_TRAJ_SEL
    # Theoretical fixation probability: 2s for Ns >> 1, 1/(2N) for neutral
    theory = 2*s if s > 0 else 1/(2*N_sel)
    print(f's={s:.2f}: simulated fixation prob={fixation_probs[s]:.3f}, theory={theory:.3f}')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Panel A: Wright-Fisher trajectories, N=20 (strong drift)
ax = axes[0, 0]
for traj in trajectories[20]:
    ax.plot(traj, alpha=0.5, lw=1, color='steelblue')
ax.axhline(0.5, color='k', ls='--', lw=1)
ax.set_xlabel('Generation'); ax.set_ylabel('Allele frequency p')
ax.set_title(f'A. Wright-Fisher, N=20\n(strong drift, p₀=0.5)')
ax.set_ylim([-0.02, 1.02])

# Panel B: Wright-Fisher trajectories, N=100
ax = axes[0, 1]
for traj in trajectories[100]:
    ax.plot(traj, alpha=0.5, lw=1, color='steelblue')
ax.axhline(0.5, color='k', ls='--', lw=1)
ax.set_xlabel('Generation'); ax.set_ylabel('Allele frequency p')
ax.set_title(f'B. Wright-Fisher, N=100\n(moderate drift)')
ax.set_ylim([-0.02, 1.02])

# Panel C: Allele frequency distribution over time (N=100)
ax = axes[0, 2]
all_trajs_100 = np.array(trajectories[100])  # (N_TRAJ, N_GEN+1)
for t_idx, col in [(0, 'lightblue'), (20, 'steelblue'), (50, 'navy'), (100, 'purple'), (200, 'tomato')]:
    freqs = all_trajs_100[:, t_idx]
    freqs = freqs[(freqs > 0) & (freqs < 1)]  # exclude fixed
    if len(freqs) > 0:
        ax.hist(freqs, bins=15, density=True, alpha=0.5, color=col, label=f't={t_idx}')
ax.set_xlabel('Allele frequency p'); ax.set_ylabel('Density')
ax.set_title('C. Allele frequency distributions\nover time (N=100)')
ax.legend(fontsize=7)

# Panel D: Selection — single trajectory with and without selection
ax = axes[1, 0]
rng2 = np.random.default_rng(99)
for s_v, col, lbl in [(0.0, 'steelblue', 'Neutral (s=0)'),
                        (0.02, 'green', 'Beneficial (s=0.02)'),
                        (-0.02, 'tomato', 'Deleterious (s=-0.02)')]:
    traj = wright_fisher(0.05, 500, 300, s=s_v, rng=rng2)
    ax.plot(traj, color=col, lw=2, label=lbl)
ax.set_xlabel('Generation'); ax.set_ylabel('Allele frequency')
ax.set_title('D. Selection vs. drift (N=500, p₀=0.05)\nsingle trajectory example')
ax.legend(fontsize=8)

# Panel E: Fixation probability vs. selection coefficient
ax = axes[1, 1]
s_theory = np.linspace(0, 0.15, 100)
N_fp = 200
fp_theory = 2 * s_theory  # approximation for Ns >> 1
fp_theory[0] = 1 / (2*N_fp)  # neutral
ax.plot(s_theory, fp_theory, 'k--', lw=1.5, label='Theory (2s, or 1/2N neutral)')
ax.scatter(list(fixation_probs.keys()), list(fixation_probs.values()),
             color='tomato', s=80, zorder=5, label='Simulated')
ax.set_xlabel('Selection coefficient s'); ax.set_ylabel('Fixation probability')
ax.set_title('E. Fixation probability vs. selection\n(starting from single copy)')
ax.legend(fontsize=8)

# Panel F: Hardy-Weinberg check diagram
ax = axes[1, 2]
ax.axis('off')
p_vals = np.linspace(0, 1, 100)
q_vals = 1 - p_vals
ax_inner = fig.add_axes([0.68, 0.12, 0.28, 0.35])
ax_inner.plot(p_vals, p_vals**2, 'steelblue', lw=2, label='AA (p²)')
ax_inner.plot(p_vals, 2*p_vals*q_vals, 'green', lw=2, label='Aa (2pq)')
ax_inner.plot(p_vals, q_vals**2, 'tomato', lw=2, label='aa (q²)')
ax_inner.set_xlabel('p (freq. of A)'); ax_inner.set_ylabel('Genotype frequency')
ax_inner.set_title('F. Hardy-Weinberg genotype frequencies', fontsize=9)
ax_inner.legend(fontsize=7)

plt.suptitle('Module 15 NB06: Population Dynamics and Evolutionary Simulation', fontsize=12, fontweight='bold')
plt.tight_layout(rect=[0,0,1,0.97])
plt.savefig('population_dynamics_evolution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Implement the Moran model: at each step, one individual is chosen to reproduce
   (proportional to fitness) and one is chosen to die (uniformly). Show that the
   fixation probability from a single copy is 1/(2N) for the neutral case —
   same as Wright-Fisher but with different fixation time distribution.
2. Implement a bottleneck: start with N=1000, run for 100 generations, reduce to
   N=20 for 20 generations, then restore N=1000. Show how heterozygosity
   (2pq) changes before, during, and after the bottleneck.
3. Simulate a selective sweep: start with a beneficial allele at frequency 1/(2N)
   and selection s=0.05. Track the linked neutral allele (hitchhiking). Show
   that after fixation, the hitchhiker allele is also at high frequency.

---
## Step 10 — Quiz

1. What is the Hardy-Weinberg principle and what are its five assumptions?
2. In the Wright-Fisher model, why does genetic drift dominate in small populations?
   Show the relationship between N and the variance of allele frequency change.
3. What is the fixation probability of a neutral allele starting from a single
   copy in a diploid population of size N? Derive it.
4. Define selection coefficient s. What does s=0.01 mean biologically?
   When does selection overpower drift?

---
## Step 12 — Reflection

> *[In one paragraph: Kimura's neutral theory claims most molecular evolution is
> driven by drift, not selection. How does the Wright-Fisher model support this
> claim? What evidence from modern genome sequencing could be used to test whether
> a specific DNA substitution was driven by drift or positive selection?]*

---
*Next: `07_network_epidemic_dynamics.ipynb`*